In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import easyocr

ラウンド中かどうかの仮判定から決めたラウンド数と実際にUIのOCRでラウンド数を比べて妥当性を検証

In [3]:
vod_title = "M8 vs. EDG - VALORANT Masters Santiago - SWISS"
video_path = Path(f"../../data/vods/{vod_title}.mp4")
labels_path = Path(f"../../data/processed/round_labels/{vod_title}_round_labels.csv")
checked_path = Path(f"../../data/processed/round_labels/{vod_title}_round_labels_checked.csv")
checked_pd = pd.read_csv(checked_path)
df = pd.read_csv(labels_path)
reader = easyocr.Reader(["en"],gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [4]:
def read_frame_at_sec(video_path, t_sec):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_idx = int(t_sec * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    ret, frame = cap.read()
    cap.release()

    if not ret:
        raise ValueError(f"Failed to read frame at {t_sec} sec")

    return frame

def crop_round_region(frame):
    h, w = frame.shape[:2]

    x1 = int(w * 0.51)
    x2 = int(w * 0.527)
    y1 = int(h * 0.01)
    y2 = int(h * 0.025)

    return frame[y1:y2, x1:x2]

def preprocess_round_number_for_ocr(img, scale=3):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    gray = cv2.resize(
        gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    _, th_fixed = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)

    _, th_otsu = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    th_adaptive = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11, 2
    )

    return {
        "gray": gray,
        "fixed": th_fixed,
        "otsu": th_otsu,
        "adaptive": th_adaptive,
    }

def read_round_number_from_crop(crop, reader):
    processed_imgs = preprocess_round_number_for_ocr(crop)

    for name, img in processed_imgs.items():
        results = reader.readtext(
            img,
            detail=0,
            allowlist="0123456789"
        )

        if len(results) == 0:
            continue

        text = "".join(results).strip()

        if text.isdigit():
            n = int(text)
            if 1 <= n <= 45:
                return n

    return None

def read_round_number_at_sec(video_path, t_sec, reader):
    frame = read_frame_at_sec(video_path, t_sec)
    crop = crop_round_region(frame)
    return read_round_number_from_crop(crop, reader)

def read_round_numbers_near_time(video_path, t_sec, reader, offsets=(-3, 0, 3)):
    results = []

    for offset in offsets:
        t = t_sec + offset

        try:
            round_no = read_round_number_at_sec(video_path, t, reader)
        except Exception as e:
            round_no = None

        results.append({
            "t_sec": t,
            "offset": offset,
            "round_no": round_no,
        })

    return results

In [5]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df)

,value,start_sec,end_sec,n_samples,mean_diff,min_diff,max_diff,duration_sec,t_sec,t_min,left_score,right_score,left_score_candidates,right_score_candidates,score_total,round_no_from_score,map_no,round_no,left_win_label
0,True,9.009,78.078,70,22.858459,17.960205,26.488715,70.070,9.009,0.150150,0,0,"[0, 0, 0]","[0, 0, 0]",0,1,1,1,1.0
1,True,106.106,203.203,98,23.621082,19.780490,27.049723,98.098,106.106,1.768433,1,0,"[1, 1, 1]","[0, 0, 0]",1,2,1,2,0.0
2,True,235.235,278.278,44,25.698283,21.291341,27.700168,44.044,235.235,3.920583,1,1,"[1, 1, 1]","[1, 1, 1]",2,3,1,3,0.0
3,True,297.297,390.390,94,25.161743,19.347548,27.986844,94.094,297.297,4.954950,1,2,"[1, 1, 1]","[2, 2, 2]",3,4,1,4,1.0
4,True,421.421,515.515,95,23.683746,19.510579,27.125407,95.095,421.421,7.023683,2,2,"[2, 2, 2]","[2, 2, 2]",4,5,1,5,0.0
5,True,542.542,650.650,109,23.825269,19.197130,27.194607,109.109,542.542,9.042367,2,3,"[2, 2, 2]","[3, 3, 3]",5,6,1,6,0.0
6,True,736.736,862.862,127,24.199966,18.810683,27.099338,127.127,736.736,12.278933,2,4,"[2, 2, 2]","[4, 4, 4]",6,7,1,7,0.0
7,True,895.895,972.972,78,25.151095,18.837484,26.630968,78.078,895.895,14.931583,2,5,"[2, 2, 2]","[5, 5, 5]",7,8,1,8,0.0
8,True,991.991,1148.147,157,23.723775,18.561252,27.512126,157.157,991.991,16.533183,2,6,"[2, 2, 2]","[6, 6, 6]",8,9,1,9,0.0
9,True,1170.169,1262.261,93,21.988577,18.152751,26.461480,93.093,1170.169,19.502817,2,7,"[2, 2, 2]","[7, 7, 7]",9,10,1,10,1.0


In [ ]:
start_offsets = [1,2,3,5]
end_offsets = [-5,-3,-2,-1]
df["round_ocr_start_samples"] = df["start_sec"].apply(
    lambda t: read_round_numbers_near_time(video_path, t, reader,offsets=start_offsets)
)
df["round_ocr_end_samples"] = df["end_sec"].apply(
    lambda t: read_round_numbers_near_time(video_path, t, reader,offsets=end_offsets)
)


c:\Users\ryo16\Desktop\valorant-\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [14]:
from collections import Counter

def majority_round(samples):
    values = [s["round_no"] for s in samples if s["round_no"] is not None]
    if not values:
        return None
    return Counter(values).most_common(1)[0][0]

df["round_ocr_start"] = df["round_ocr_start_samples"].apply(majority_round)
df["round_ocr_end"] = df["round_ocr_end_samples"].apply(majority_round)
df["round_check_start"] = df["round_ocr_start"] == df["round_no"]
df["round_check_end"] = df["round_ocr_end"] == df["round_no"]
bad_end = df[df["round_check_end"] == False]
bad_start = df[df["round_check_end"] == False]


In [21]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df[["map_no","start_sec","end_sec", "round_no","round_ocr_start", "round_ocr_end"]])

,map_no,start_sec,end_sec,round_no,round_ocr_start,round_ocr_end
0,1,9.009,78.078,1,1,1
1,1,106.106,203.203,2,2,2
2,1,235.235,278.278,3,3,3
3,1,297.297,390.390,4,4,4
4,1,421.421,515.515,5,5,5
5,1,542.542,650.650,6,6,6
6,1,736.736,862.862,7,7,7
7,1,895.895,972.972,8,8,8
8,1,991.991,1148.147,9,9,9
9,1,1170.169,1262.261,10,10,10
